# Viveka — GRPO Training: Qwen2.5-14B-Instruct on Kaggle (OOM-mitigated)

Train **Qwen/Qwen2.5-14B-Instruct** with TRL GRPO + Unsloth 4-bit QLoRA. **This run is exploratory** — the handoff notes from the 2026-05-25 session explicitly rejected 14B for OOM on free T4 with the default `G=4` GRPO config. This notebook tries again with aggressive mitigations applied as an inline patch to `train.py`:

| Knob | Default | This run | Why |
|---|---|---|---|
| `num_generations` | 4 | **2** | Halves rollout memory; group-relative advantage still computable |
| `gradient_accumulation_steps` | 4 | **8** | Keeps effective batch size at 8 (matches v6 hyperparams) |
| `max_seq` | 1280 | **1024** | Cuts per-step activation memory by ~20% |
| LoRA `r` | 16 | **8** | Halves adapter footprint + optimizer state |

If the smoke run still OOMs (look for `torch.cuda.OutOfMemoryError`), kill the run and either (a) try Qwen-2.5-3B-Instruct as a fallback by changing `--model` and skipping the patch cell, or (b) move to a paid Kaggle hardware tier.

**Why attempt at all:** the 7B → 14B jump within the Qwen family is the steepest capacity gradient available without leaving T4-class compute. If Qwen-7B passes T4 traps and 14B fails, that's a "more capacity makes it worse" signal — close kin to MacDiarmid et al. Nov 2025 (Anthropic emergent misalignment at production scale).

**TRL EOS coverage:** identical to 7B / 1.5B — `<|im_end|>` (151645) + `<|endoftext|>` (151643). In-repo fix at `train.py:531-595` handles it.

## Prereqs (do these BEFORE running)

1. Notebook **Settings**: Accelerator = `GPU T4 x2`, Internet = `On`, Persistence = `Files only`
2. **Add-ons → Secrets**: add `HF_TOKEN` (HuggingFace **write** token)
3. Run cells **in order**. The patch cell (Cell 5) modifies `train.py` in-place — re-clone if you need to revert.

Total wall time: ~25–45 min smoke (if it doesn't OOM), ~8–12 hours full run on T4 x2.

## Model facts (as of 2026)

| Field | Value |
|---|---|
| Parameters | 14.7B |
| Architecture | Qwen2 (48 layers, 40 q-heads, 8 kv-heads, hidden 5120) |
| Chat template | ChatML (`<|im_start|>` / `<|im_end|>`) |
| Tokenizer EOS | `<|im_end|>` (id 151645) |
| `generation_config.eos_token_id` | `[151645, 151643]` — same TRL bug class as Qwen-1.5B / 7B |
| 4-bit weight footprint | ~9 GB |
| GRPO config (patched below) | G=2, max_seq=1024, LoRA r=8, grad accum=8 |


In [ ]:
# Step 1: GPU check + clone HF Space + tokens ───────────────────────
import os
from kaggle_secrets import UserSecretsClient

# Move to a known-good parent BEFORE any rm/clone — if the kernel was
# previously inside /kaggle/working/viveka-env and we rm -rf it, the
# shell loses its CWD and every subsequent command fails.
os.chdir("/")
os.chdir("/kaggle/working")
%cd /kaggle/working

!nvidia-smi

# Clone from the HF Space (public; no GitHub token needed).
# Space mirror of github.com/DevMhrn/viveka-env, pushed 2026-05-25.
!rm -rf /kaggle/working/viveka-env
!git lfs install --skip-repo 2>/dev/null || true
!git clone https://huggingface.co/spaces/ddevMhrn/viveka-env /kaggle/working/viveka-env
%cd /kaggle/working/viveka-env

# HF_TOKEN needed for:
#   1. Downloading the base model (Llama-3.1 is gated; Qwen is open)
#   2. Pushing the trained LoRA adapter to ddevMhrn/<model>-Viveka at end
# Token must have WRITE scope. Store in Kaggle Add-ons → Secrets as HF_TOKEN.
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

!git log --oneline -5


## Install — pin every layer that's bitten us

Order matters. Each pin solves a specific known issue:
- `openenv-core==0.2.2` (0.2.3 wheel has stale fastmcp imports)
- `fastmcp==3.1.1` (3.2 removed `CallToolResult`, 2.x is wrong era)
- `mcp` upgrade (Kaggle's preinstalled is too old, missing `Icon` class)
- `trl>=0.13` (GRPOConfig added in 0.13; we use the `generation_kwargs` override added in trl#3562 to dodge the Qwen/Llama multi-EOS bug)
- `unsloth` (4-bit QLoRA training)
- `bitsandbytes` (4-bit quantisation backend)
- `weave` (TRL ≥ 0.16 imports it inside `grpo_trainer`; subprocess needs the real package)
- `huggingface_hub[cli]` (for pushing LoRA at the end)


In [ ]:
# Step 2: Install all deps in known-good order ──────────────────────

# Step 1: project itself in editable mode (uses pyproject pins)
!pip install -q -e ".[train]"

# Step 2: force-pin the openenv-core / fastmcp pair that works
!pip install --upgrade --force-reinstall --no-deps "openenv-core==0.2.2" "fastmcp==3.1.1"

# Step 3: upgrade mcp (Kaggle's preinstalled mcp lacks Icon class — fastmcp 3.1.1 imports it)
!pip install -q -U "mcp"

# Step 4: openenv-core needs uncalled-for transitively
!pip install -q -U "uncalled-for"

# Step 5: TRL >= 0.13 for GRPOConfig
!pip install -q -U "trl>=0.13.0"

# Step 6: mergekit (TRL transitive)
!pip install -q mergekit

# Step 7: Unsloth (Kaggle/Colab compatible)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Step 8: fresh bitsandbytes for 4-bit quant
!pip install -q -U bitsandbytes

# Step 9: weave — W&B tracing library; TRL >= 0.16 imports it transitively
# from trl.trainer.grpo_trainer. Without this, `from trl import GRPOTrainer`
# raises ModuleNotFoundError. Stubbing isn't enough — train.py runs in a
# subprocess and needs the real package on the Python path.
!pip install -q weave

# Step 10: Remove torchao. Kaggle preinstalls torchao 0.10.0, but peft >= 0.14
# calls is_torchao_available() which RAISES ImportError when an installed
# torchao is older than 0.16.0 (instead of returning False cleanly). This
# bubbles up through PeftModel.from_pretrained() and aborts the inference
# eval steps (Steps 10-11) that load the trained LoRA. Training itself is
# unaffected (Unsloth's FastLanguageModel.get_peft_model bypasses this
# dispatcher chain), so the bug only surfaces when the notebook reaches
# its sealed-eval cells. We don't use torchao anywhere — bnb-4bit is the
# quant path — so removing it makes the dispatcher skip cleanly.
!pip uninstall -y torchao 2>&1 | tail -2

# Step 11: HF Hub CLI (for LoRA push at end)
!pip install -q -U "huggingface_hub[cli]"

print("\n=== installed versions ===")
!pip show openenv-core fastmcp mcp trl transformers huggingface-hub peft uncalled-for unsloth bitsandbytes 2>&1 | grep -E "^(Name|Version)" 


In [ ]:
# Step 3: Verify all critical imports clean ─────────────────────────
# Order matters: stubs first (so trl's lazy imports succeed), then unsloth
# (so it can patch transformers/trl/peft BEFORE they're loaded by test()),
# then everything else.
import importlib, importlib.util, sys, types

# ─── 1) Stubs for broken transitive imports (must precede trl) ─────────
class _DummyModule(types.ModuleType):
    def __getattr__(self, name):
        if name.startswith("__"):
            raise AttributeError(name)
        return type(name, (), {})

def _install_stub(modname, dummy=False):
    if modname in sys.modules:
        return
    m = _DummyModule(modname) if dummy else types.ModuleType(modname)
    m.__spec__ = importlib.util.spec_from_loader(modname, None)
    sys.modules[modname] = m

# llm_blender — broken on transformers 4.45+
try:
    import llm_blender  # noqa: F401
except Exception:
    _install_stub("llm_blender")
    sys.modules["llm_blender"].Blender = type("Blender", (), {})

# mergekit — pydantic schema crash on its torch.Tensor field
for _name in ["mergekit", "mergekit.merge_methods", "mergekit.io",
              "mergekit.config", "mergekit.architecture", "mergekit.options",
              "mergekit.merge", "mergekit.plan", "mergekit.graph"]:
    _install_stub(_name, dummy=True)

print("shims applied")

# ─── 2) Import Unsloth FIRST, before anything transformers/trl/peft ───
# Unsloth monkey-patches those packages at import time; if they're already
# loaded, patches silently skip and you lose the 2x speedup. Suppress the
# benign UserWarning unsloth emits about import order — by importing it
# here, we ARE doing what it asks.
unsloth_ok = False
try:
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        from unsloth import FastLanguageModel  # noqa: F401
    print("\u2705 unsloth.FastLanguageModel (loaded first, patches armed)")
    unsloth_ok = True
except Exception as e:
    print(f"\u274c unsloth: {type(e).__name__}: {e}")

# ─── 3) Now the rest can safely import trl/openenv/viveka ─────────────
def test(module):
    try:
        importlib.import_module(module)
        print(f"\u2705 {module}")
        return True
    except Exception as e:
        print(f"\u274c {module}: {type(e).__name__}: {e}")
        return False

ok = unsloth_ok
ok &= test("trl")
ok &= test("openenv.core.rubrics.trajectory")
ok &= test("openenv.core.env_server.types")
ok &= test("openenv.core.env_server.http_server")

try:
    from trl import GRPOConfig, GRPOTrainer  # noqa: F401
    print("\u2705 trl.GRPOConfig + GRPOTrainer")
except Exception as e:
    print(f"\u274c GRPO: {type(e).__name__}: {e}"); ok = False

try:
    sys.path.insert(0, "/kaggle/working/viveka-env")
    from viveka.server.environment import VivekaEnvironment  # noqa: F401
    print("\u2705 viveka.server.environment.VivekaEnvironment")
except Exception as e:
    print(f"\u274c viveka env: {type(e).__name__}: {e}"); ok = False

print(f"\n{'\u2705 ALL CLEAN — proceed' if ok else '\u274c FIX BEFORE PROCEEDING'}")


In [ ]:
# Step 4: Dry-run (no GPU touch, ~10 sec) ───────────────────────────
# Validates train.py builds the env + dataset + scoring path without loading the model.
!python train.py --dry-run --model Qwen/Qwen2.5-14B-Instruct


In [ ]:
# Step 4b: PATCH train.py for OOM mitigation (14B-specific) ─────────
# Knobs reduced vs default train.py:
#   num_generations 4 -> 2
#   gradient_accumulation_steps 4 -> 8
#   max_seq = 1280 -> 1024
#   LoRA r=16 -> r=8
#
# Patch is idempotent (uses unique anchor strings); re-running is a no-op.
# To revert: re-run Step 1 (clone) to restore train.py.

from pathlib import Path

train_py = Path("/kaggle/working/viveka-env/train.py")
src = train_py.read_text()

patches = [
    ("max_seq = 1280", "max_seq = 1024"),
    ("        r=16,", "        r=8,"),
    ("        per_device_train_batch_size=1,\n        gradient_accumulation_steps=4,",
     "        per_device_train_batch_size=1,\n        gradient_accumulation_steps=8,"),
    ("        num_generations=4,", "        num_generations=2,"),
]

applied = []
for old, new in patches:
    if old in src:
        src = src.replace(old, new, 1)
        applied.append(old.strip().split("=")[0].strip() if "=" in old else old[:30])
    else:
        # check if already patched (idempotent re-run)
        if new in src:
            applied.append(f"{old.strip()[:30]} (already patched)")
        else:
            print(f"\u26a0\ufe0f  anchor not found, skipping: {old!r}")

train_py.write_text(src)

print("Applied patches:")
for a in applied:
    print(f"  - {a}")

# Verify patched values landed
print("\n=== verification ===")
!grep -n "max_seq\s*=\s*\|r=8,\|r=16,\|num_generations=\|gradient_accumulation_steps=" train.py | head -10


## Smoke run (10 episodes, ~15–25 min)

Watch for:
1. `[load] Qwen2.5-14B via Unsloth 4-bit ...`
2. `[fix] generation_kwargs.eos_token_id list = [...]` — confirms TRL EOS workaround armed
3. Training step logs with `loss` and `grad_norm`
4. **No `[NaNGuard]` halts**
5. Final `[smoke] terminal reward=...` line

If smoke crashes — STOP, paste the error, do NOT run the full training cell.


In [ ]:
# Step 5: Smoke run (10 episodes) ───────────────────────────────────
!python train.py --smoke \
    --model Qwen/Qwen2.5-14B-Instruct \
    --output-dir /kaggle/working/runs/smoke \
    --no-wandb


## Full training: 200 episodes, ~8–12 hours on T4

DO NOT RUN until smoke (Cell 5) finishes cleanly.

While this runs:
- Sample 5 generations every 30 min — look for reward hacking / spec gaming
- If reward stays flat for >30 steps, kill the run and surface in main session
- Tier mix `1:0.4, 2:0.4, 4:0.2` matches the v6 (Qwen-1.5B) training run

Trained LoRA adapter lands at `/kaggle/working/runs/Qwen2.5-14B_v1/lora/`.


In [ ]:
# Step 6: Full training (200 episodes) ──────────────────────────────
!python train.py \
    --model Qwen/Qwen2.5-14B-Instruct \
    --episodes 200 \
    --output-dir /kaggle/working/runs/qwen25_14b_v1 \
    --tier-mix "1:0.4,2:0.4,4:0.2" \
    --no-wandb


In [ ]:
# Step 7: Generate reward curve from training log ───────────────────
# reward_curve.py requires: --training-log, --baseline-json, --output-png.
# Baseline JSONs live in eval/results/ in the cloned repo. baseline_random.json
# is the random-policy mean reward across the 68 scenarios; using it as the
# horizontal reference shows how far above random the trained model climbed.
RUN_DIR = "/kaggle/working/runs/qwen25_14b_v1"

!python eval/reward_curve.py \
    --training-log $RUN_DIR/training_log.jsonl \
    --baseline-json eval/results/baseline_random.json \
    --output-png $RUN_DIR/reward_curve.png \
    --smooth-window 10 \
    --title "GRPO Training — qwen25 14b v1 (200 episodes)" 2>&1 | tail -5

!ls -la $RUN_DIR/


## Sealed eval — base vs trained, T1+T2 then T3+T4

Matches the published pattern in `eval/results/` (`llama1b_*`, `llama3b_*` etc.): four `inference.py` passes per model, split into `t12` and `t34` because the harder tiers take longer per scenario and the split fits Kaggle session limits.

`--per-tier 25` is larger than any tier's scenario count (max is T2=20), so each pass pulls **all** scenarios from the requested tiers — full 68-scenario coverage across the four logs. Each log ends with a SUMMARY block (mean reward, per-tier breakdown, termination distribution, action-type histogram) that's the publishable artifact.

**T3+T4 trained log is the showcase number** — its T4 row tells you how many `must_not_execute` traps fired. That's the load-bearing claim in the blog narrative.


In [ ]:
# Step 8: Inference — FROZEN base on T1+T2 ─────────────────────────
# Apples-to-apples "untrained" comparison. Same prompts, same env, no LoRA.
RUN_DIR = "/kaggle/working/runs/qwen25_14b_v1"

!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen \
    --model Qwen/Qwen2.5-14B-Instruct \
    --tier-mix 1,2 \
    --per-tier 25 \
    --output-json $RUN_DIR/qwen14b_base_t12.json \
    2>&1 | tee $RUN_DIR/qwen14b_base_t12.log


In [ ]:
# Step 9: Inference — FROZEN base on T3+T4 ─────────────────────────
RUN_DIR = "/kaggle/working/runs/qwen25_14b_v1"

!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen \
    --model Qwen/Qwen2.5-14B-Instruct \
    --tier-mix 3,4 \
    --per-tier 25 \
    --output-json $RUN_DIR/qwen14b_base_t34.json \
    2>&1 | tee $RUN_DIR/qwen14b_base_t34.log


In [ ]:
# Step 10: Inference — TRAINED (base + LoRA) on T1+T2 ──────────────
RUN_DIR = "/kaggle/working/runs/qwen25_14b_v1"

!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen \
    --model Qwen/Qwen2.5-14B-Instruct \
    --adapter $RUN_DIR/lora \
    --tier-mix 1,2 \
    --per-tier 25 \
    --output-json $RUN_DIR/qwen14b_train_t12.json \
    2>&1 | tee $RUN_DIR/qwen14b_train_t12.log


In [ ]:
# Step 11: Inference — TRAINED on T3+T4 (the showcase tier) ────────
RUN_DIR = "/kaggle/working/runs/qwen25_14b_v1"

!cd /kaggle/working/viveka-env && CUDA_VISIBLE_DEVICES=0 python inference.py \
    --policy qwen \
    --model Qwen/Qwen2.5-14B-Instruct \
    --adapter $RUN_DIR/lora \
    --tier-mix 3,4 \
    --per-tier 25 \
    --output-json $RUN_DIR/qwen14b_train_t34.json \
    2>&1 | tee $RUN_DIR/qwen14b_train_t34.log

print("\n=== compare base vs trained ===")
!grep -A 6 "SUMMARY" $RUN_DIR/qwen14b_*.log


## Push trained LoRA + eval results to HF Hub: `ddevMhrn/Qwen2.5-14B-Viveka`

Creates the repo if it doesn't exist (idempotent), copies all eval artifacts into the LoRA folder, then uploads the lot. Requires the Kaggle `HF_TOKEN` secret to have **write** scope — verify at `https://huggingface.co/settings/tokens` before running.

After this cell completes, everything for this model lives at `huggingface.co/ddevMhrn/Qwen2.5-14B-Viveka`:

```
adapter_config.json, adapter_model.safetensors  (the LoRA)
tokenizer files
training_log.jsonl     (per-step metrics)
reward_curve.png       (training reward chart)
holdout_quick.{json,md}   (15-scenario quick eval)
holdout_full.{json,md}    (68-scenario full eval — if Step 9 ran)
README.md              (model card)
```


In [ ]:
# Step 12: Push LoRA + eval artifacts to HF Hub ────────────────────
import os, json, shutil
from pathlib import Path
from huggingface_hub import HfApi, create_repo

REPO_ID = "ddevMhrn/Qwen2.5-14B-Viveka"
RUN_DIR = Path("/kaggle/working/runs/qwen25_14b_v1")
LORA_DIR = RUN_DIR / "lora"

assert LORA_DIR.exists(), f"missing LoRA dir: {LORA_DIR}"

# Bundle everything into the LoRA folder so one upload ships the lot.
# Each file is optional — copy only what's present from the run.
for fname in [
    "training_log.jsonl",                     # per-step training metrics
    "reward_curve.png",                        # Step 7 output
    "qwen14b_base_t12.log",   "qwen14b_base_t12.json",   # Step 8
    "qwen14b_base_t34.log",   "qwen14b_base_t34.json",   # Step 9
    "qwen14b_train_t12.log",  "qwen14b_train_t12.json",  # Step 10
    "qwen14b_train_t34.log",  "qwen14b_train_t34.json",  # Step 11
]:
    src = RUN_DIR / fname
    if src.exists():
        shutil.copy(src, LORA_DIR / fname)
        print(f"  bundled {fname}")

# Write a model card README
card = f"""---
library_name: peft
base_model: Qwen/Qwen2.5-14B-Instruct
license: apache-2.0
tags:
  - viveka
  - grpo
  - reversibility
  - calibrated-confidence
  - indic-dpi
  - openenv
---

# Qwen2.5-14B-Viveka

LoRA adapter trained on the [Viveka OpenEnv](https://huggingface.co/spaces/ddevMhrn/viveka-env) with TRL GRPO + Unsloth 4-bit QLoRA. Six-component deterministic reward over mocked Indian DPI services (UPI, DigiLocker, IRCTC). 200 episodes, tier mix 1:0.4 / 2:0.4 / 4:0.2.

**Base model:** `Qwen/Qwen2.5-14B-Instruct`

**Notes:** OOM-mitigated config: G=2, max_seq=1024, LoRA r=8, grad accum=8. Halved generations and adapter rank vs v6; downstream comparison should weight that.

See [github.com/DevMhrn/viveka-env](https://github.com/DevMhrn/viveka-env) for the env, reward design, and eval harness.
"""
(LORA_DIR / "README.md").write_text(card)

# Create repo + upload
create_repo(REPO_ID, repo_type="model", exist_ok=True, private=False, token=os.environ["HF_TOKEN"])
api = HfApi()
api.upload_folder(
    folder_path=str(LORA_DIR),
    repo_id=REPO_ID,
    repo_type="model",
    token=os.environ["HF_TOKEN"],
    commit_message="feat(training): GRPO LoRA from Kaggle run",
)
print(f"\n\u2705 pushed to https://huggingface.co/{REPO_ID}")
